> 🔑 **교수자용 답안 노트북** — 학생용 파일: `unit1_python.ipynb`

# 1단원 파이썬 확장 실습
**디지털리터러시 36시간 과정 · 인공지능 시대의 디지털 리터러시**

| 파트 | 연결 차시 | 내용 |
|---|---|---|
| A | 5차시 AI 원리 | "다음 단어 맞히기" 모델 직접 만들기 |
| B | 7차시 보안 | 안전한 비밀번호 생성기 · 강도 검사기 |

### 사용법
- 셀을 클릭하고 **Shift + Enter** → 실행하고 다음 셀로 이동합니다. **위에서부터 순서대로** 실행하세요.
- **✏️ 셀**: 빈칸(`TODO`)을 직접 채웁니다. 바로 아래 확인 셀에서 ✅가 나오면 성공입니다.
- **🤖 표시**: ChatGPT나 Gemini에게 물어보는 단계입니다. AI가 준 코드는 **반드시 실행해서 확인**합니다.
- 오류가 나도 괜찮습니다. 오류 메시지의 **마지막 줄**을 읽고, 그래도 모르겠으면 그 줄을 AI에게 붙여 넣어 물어보세요.
- 개인정보(실제 비밀번호, 주민등록번호, 전화번호)는 입력하지 않습니다.

In [ ]:
# 정답 확인 도우미 — 먼저 한 번 실행하세요
def 확인(이름, 결과, 정답):
    if 결과 is None:
        print(f"⏳ {이름}: 아직 비어 있어요 — ✏️ 셀을 채우고 다시 실행하세요")
    elif 결과 == 정답:
        print(f"✅ {이름}: 정답입니다!")
    else:
        print(f"❌ {이름}: 결과 {결과} / 기대 {정답} — 코드를 다시 확인하세요")

print("준비 완료")

---
## A. "다음 단어 맞히기" 모델 만들기 (5차시)

ChatGPT 같은 생성형 AI는 **"지금까지의 글 다음에 올 말로 가장 그럴듯한 것"** 을 계속 고르면서 문장을 만듭니다.
실제 AI는 인터넷 규모의 글과 복잡한 신경망을 쓰지만, **원리의 뼈대**는 우리가 딕셔너리로 만들 수 있습니다.

### A-1. 학습 데이터 준비 — 문장을 단어로 쪼개기

In [ ]:
문장들 = [
    "나는 오늘 아침에 커피를 마셨다",
    "나는 오늘 점심에 김밥을 먹었다",
    "나는 오늘 저녁에 친구를 만났다",
    "나는 어제 아침에 운동을 했다",
    "오늘 아침에 비가 왔다",
    "오늘 점심에 회의가 있었다",
    "친구를 만나서 커피를 마셨다",
]

for 문장 in 문장들:
    print(문장.split())

### A-2. ✏️ "다음 단어 표" 만들기
각 단어 다음에 어떤 단어가 **몇 번** 나왔는지 셉니다. 결과는 이런 모양의 딕셔너리입니다.

```
{ "나는": {"오늘": 3, "어제": 1},
  "오늘": {"아침에": 2, "점심에": 2, "저녁에": 1}, ... }
```
**TODO**: `표[지금]` 안에서 `다음` 단어의 횟수를 1 늘리세요.
힌트: `표`에 `지금`이 아직 없으면 빈 딕셔너리 `{}`를 먼저 넣습니다. 딕셔너리의 `get(키, 기본값)`이 편합니다.

In [ ]:
def 다음단어_표_만들기(문장들):
    표 = {}
    for 문장 in 문장들:
        단어들 = 문장.split()
        for i in range(len(단어들) - 1):
            지금 = 단어들[i]
            다음 = 단어들[i + 1]
            if 지금 not in 표:
                표[지금] = {}
            표[지금][다음] = 표[지금].get(다음, 0) + 1
    return 표

표 = 다음단어_표_만들기(문장들)
print(표)

In [ ]:
# 확인 셀
확인('"나는" 다음 단어', 표.get("나는"), {"오늘": 3, "어제": 1})
확인('"오늘" 다음 단어', 표.get("오늘"), {"아침에": 2, "점심에": 2, "저녁에": 1})

### A-3. 다음 단어 확률 보기
횟수를 전체 합으로 나누면 **확률**이 됩니다. 실제 AI도 다음 단어 후보마다 이런 확률을 계산합니다.

In [ ]:
def 다음단어_확률(표, 단어):
    if 단어 not in 표:
        print(f"'{단어}' 다음에 올 단어를 배운 적이 없어요.")
        return {}
    후보 = 표[단어]
    전체 = sum(후보.values())
    확률 = {}
    for 다음, 횟수 in sorted(후보.items(), key=lambda x: -x[1]):
        확률[다음] = 횟수 / 전체
        print(f"  {단어} → {다음:<6} {확률[다음] * 100:5.1f}%  " + "■" * round(확률[다음] * 20))
    return 확률

다음단어_확률(표, "나는")
print()
다음단어_확률(표, "오늘")

### A-4. 문장 만들기 — 실행할 때마다 달라질까?
- `무작위=False`: 항상 **가장 확률이 높은** 단어를 고릅니다.
- `무작위=True`: 확률에 **비례해서 뽑습니다.** (실제 AI도 이렇게 약간의 무작위성을 둡니다)

👉 아래 셀을 **5번** 실행해 보세요.

In [ ]:
import random

def 문장_만들기(표, 시작, 길이=6, 무작위=True):
    결과 = [시작]
    지금 = 시작
    for _ in range(길이 - 1):
        if 지금 not in 표:
            break
        후보 = 표[지금]
        if 무작위:
            지금 = random.choices(list(후보), weights=list(후보.values()))[0]
        else:
            지금 = max(후보, key=후보.get)
        결과.append(지금)
    return " ".join(결과)

print("항상 가장 높은 것 :", 문장_만들기(표, "나는", 무작위=False))
print("확률대로 뽑기     :", 문장_만들기(표, "나는", 무작위=True))

### A-5. 데이터를 바꾸면 모델이 바뀐다
아래 셀에 문장을 **5개 이상 더** 넣고 실행한 뒤, 확률이 어떻게 달라졌는지 봅니다.
예: "나는 어제 저녁에 영화를 봤다", "나는 어제 점심에 국수를 먹었다" …

In [ ]:
추가_문장들 = [
    "나는 어제 저녁에 영화를 봤다",
    # 여기에 문장을 더 넣어 보세요
]

새_표 = 다음단어_표_만들기(문장들 + 추가_문장들)
print("[원래 데이터]")
다음단어_확률(표, "나는")
print("[문장 추가 후]")
다음단어_확률(새_표, "나는")
print()
다음단어_확률(새_표, "고양이는")   # 배운 적 없는 단어

#### 💬 생각해 보기 (실습지에 적기)
1. 같은 "나는"으로 시작해도 문장이 매번 달라진 이유는?
2. 학습 데이터에 한쪽 이야기만 많으면 결과는 어떻게 치우칠까요? → **AI의 편향**
3. 우리 모델은 "고양이는"을 모른다고 멈췄습니다. 실제 생성형 AI는 모르는 내용도 **그럴듯하게 지어낼 수 있습니다.** → 2단원에서 배울 **환각**

🤖 **AI에게 물어보기**: "위 `문장_만들기` 함수를 파이썬 기초만 배운 사람이 이해하게 한 줄씩 설명해 줘"

---
## B. 안전한 비밀번호 생성기 · 강도 검사기 (7차시)

> ⚠️ **지금 실제로 쓰는 비밀번호는 절대 입력하지 마세요.** 예시 문자열로만 시험합니다.

### B-1. 비밀번호 생성기
`random`은 게임·시뮬레이션용이라 **다음 값을 예측할 수 있는 방식**입니다. 비밀번호·인증번호처럼 보안이 필요한 곳에는 파이썬이 권장하는 `secrets` 모듈을 씁니다.

In [ ]:
import secrets
import string

특수문자 = "!@#$%^&*"

def 비밀번호_만들기(길이=14):
    문자 = string.ascii_letters + string.digits + 특수문자
    while True:
        pw = "".join(secrets.choice(문자) for _ in range(길이))
        # 소문자·대문자·숫자·특수문자가 모두 들어갔을 때만 사용
        if (any(c.islower() for c in pw) and any(c.isupper() for c in pw)
                and any(c.isdigit() for c in pw) and any(c in 특수문자 for c in pw)):
            return pw

for _ in range(3):
    print(비밀번호_만들기())

### B-2. ✏️ 강도 검사기 만들기
점수 규칙 (0~5점)

| 규칙 | 점수 |
|---|---|
| 흔한 비밀번호 목록에 있음 | **바로 0점** |
| 12자 이상 / 16자 이상 | +1 / 추가 +1 |
| 소문자와 대문자가 **섞여** 있음 | +1 |
| 숫자가 있음 | +1 |
| 특수문자(글자·숫자가 아닌 것)가 있음 | +1 |
| 같은 문자 3번 연속 (aaa, 111) | −1 |
| 연속 숫자 (123, 789) | −1 |

**TODO 4곳**을 채우세요. 힌트: `any(c.isupper() for c in pw)`, `c.isalnum()`, `pw[i] == pw[i+1] == pw[i+2]`

In [ ]:
흔한_비밀번호 = ["password", "123456", "12345678", "qwerty", "qwer1234",
               "111111", "abc123", "iloveyou", "admin", "1q2w3e4r"]

def 강도_검사(pw):
    if pw.lower() in 흔한_비밀번호:
        return 0, ["누구나 아는 흔한 비밀번호입니다"]
    점수 = 0
    조언 = []

    if len(pw) >= 12:
        점수 += 1
    else:
        조언.append("12자 이상으로 늘리세요")
    if len(pw) >= 16:
        점수 += 1

    if any(c.islower() for c in pw) and any(c.isupper() for c in pw):
        점수 += 1
    else:
        조언.append("대문자와 소문자를 섞으세요")
    if any(c.isdigit() for c in pw):
        점수 += 1
    else:
        조언.append("숫자를 넣으세요")
    if any(not c.isalnum() for c in pw):
        점수 += 1
    else:
        조언.append("특수문자를 넣으세요")

    for i in range(len(pw) - 2):
        if pw[i] == pw[i + 1] == pw[i + 2]:
            점수 -= 1
            조언.append("같은 문자를 3번 연속 쓰지 마세요")
            break

    for i in range(len(pw) - 2):
        a, b, c = pw[i], pw[i + 1], pw[i + 2]
        if a.isdigit() and b.isdigit() and c.isdigit() and int(b) == int(a) + 1 and int(c) == int(b) + 1:
            점수 -= 1
            조언.append("123처럼 연속된 숫자는 피하세요")
            break

    return max(0, min(점수, 5)), 조언

In [ ]:
# 확인 셀 — 테스트 5개
테스트 = [("password", 0), ("abcd1234", 0), ("Summer2026!", 3),
          ("coffee-tree-moon-47", 4), ("Aaaa1111!!", 2)]
통과 = 0
for pw, 기대 in 테스트:
    점수, 조언 = 강도_검사(pw)
    ok = 점수 == 기대
    통과 += ok
    print(("✅" if ok else "❌"), f"{pw:<22} {점수}점 (기대 {기대}점)", 조언)
print(f"\n{통과}/{len(테스트)} 통과")

### B-3. 비밀번호 문구(passphrase) 만들기
무작위 단어 여러 개를 이어 붙이면 **길고, 외우기 쉽고, 추측하기 어려운** 비밀번호가 됩니다.

In [ ]:
단어장 = ["바다", "연필", "구름", "사과", "기차", "우산", "나무", "별빛",
          "coffee", "river", "piano", "tiger", "maple", "cloud", "lemon", "stone"]

def 문구_만들기(개수=4):
    단어들 = [secrets.choice(단어장) for _ in range(개수)]
    숫자 = str(secrets.randbelow(90) + 10)
    return "-".join(단어들) + "-" + 숫자

for pw in [비밀번호_만들기(), 문구_만들기(), "Summer2026!"]:
    print(f"{pw:<30}", 강도_검사(pw))

### B-4. 🤖 AI에게 검증받기
아래처럼 물어보고, AI가 말한 약점을 **실제 예시 문자열로 넣어 확인**하세요.

```
나 ▶ 아래 파이썬 비밀번호 강도 검사 함수가 놓치는 약점 2가지를 알려 주고,
     그 약점 때문에 점수가 높게 잘못 나오는 예시 문자열도 하나씩 보여 줘.
     (강도_검사 함수 코드 붙여 넣기)
```

예상되는 약점: 키보드 배열(`qwerasdf`), 이름+생일 같은 개인정보, 흔한 단어에 숫자만 붙인 것 등.
👉 실제 서비스는 유출된 비밀번호 목록까지 대조하는 훨씬 정교한 검사를 씁니다. **점수보다 중요한 것은 사이트마다 다른 비밀번호 + 2단계 인증**입니다.

In [ ]:
# AI가 알려 준 약점 예시를 넣어 확인해 보세요
print(강도_검사("Qwerasdf!2024"))